# Stage 1 GPU Pipeline — Value Axis (Qwen3-8B)

This notebook runs the **GPU-heavy** Stage 1 steps on Colab:

1. Extract Qwen3-8B activations from ICRL conversations
2. Build the value axis (`value_axis.npy`)
3. Evaluate held-out AUROC and run the **gate** (L21/L22 ≥ 0.93)

**Not included here:** ICRL generation (`icrl_gen/generate.py`) — run that locally or on any machine with an Anthropic API key, then upload `icrl.json`.

**Runtime:** Colab Pro / A100 recommended. T4 (16GB) may work with bf16 + `device_map="auto"` but can OOM on long conversations.

**Before you start:**
- Runtime → **Change runtime type → GPU**
- Hugging Face account + access to [Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B)
- `icrl.json` from `python -m stage1.icrl_gen.generate` (pilot or full 300)

## 1. GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} ({mem_gb:.1f} GB)")
if mem_gb < 15:
    print("Warning: <16GB VRAM may OOM on Qwen3-8B. Prefer A100/L4 if available.")

## 2. Get the `stage1` code on this runtime

Pick **one** option below.

### Option A — Google Drive (recommended)

Copy your local `Research/stage1/` folder to Google Drive, e.g.:
`My Drive/Research/stage1/`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

STAGE1_DIR = "/content/drive/MyDrive/Research/stage1"  # <-- adjust if needed

import os
assert os.path.isdir(STAGE1_DIR), f"Folder not found: {STAGE1_DIR}"
os.chdir(STAGE1_DIR)
print("Working directory:", os.getcwd())

### Option B — Upload a zip of `stage1/`

Zip your local `stage1` folder, upload when prompted, then uncomment/run.

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # select stage1.zip
# !unzip -q -o stage1.zip -d /content
# import os
# STAGE1_DIR = "/content/stage1"
# os.chdir(STAGE1_DIR)
# print("Working directory:", os.getcwd())

## 3. Install dependencies

In [ ]:
!pip install -q -e .

import stage1
from stage1.common.paths import DATA_DIR, data_file
print("DATA_DIR:", DATA_DIR)

## 4. Hugging Face login

Required to download `Qwen/Qwen3-8B`. Use a [read token](https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import login

login()  # paste token when prompted

## 5. Provide ICRL conversations

Either upload `icrl.json` / `icrl_pilot.json`, or place it on Drive at `stage1/data/icrl.json`.

In [ ]:
from pathlib import Path
from google.colab import files

ICRL_PATH = data_file("icrl.json")

if not ICRL_PATH.exists():
    print(f"{ICRL_PATH} not found — upload icrl.json now")
    uploaded = files.upload()
    name = next(iter(uploaded))
    ICRL_PATH.parent.mkdir(parents=True, exist_ok=True)
    Path(name).rename(ICRL_PATH)

import json
with open(ICRL_PATH) as f:
    n_convs = len(json.load(f))
print(f"Using {ICRL_PATH} ({n_convs} conversations)")

## 6. Extract activations (slow — one forward pass per conversation)

Caches to `data/activations/*.npz`. Re-run with `--force` to overwrite.

In [ ]:
!python -m stage1.pipeline.extract_activations --icrl {ICRL_PATH} --force

## 7. Build axis, evaluate AUROC, run gate

Gate passes if **L21 and L22 AUROC ≥ 0.93** (paper targets: 0.954 / 0.947).

In [ ]:
!python -m stage1.pipeline.run_gate --icrl {ICRL_PATH} --skip-mock --skip-extract

## 8. Results

In [ ]:
import json
from IPython.display import Image, display

auroc_path = data_file("auroc_by_layer.json")
plot_path = data_file("auroc_by_layer.png")
axis_path = data_file("value_axis.npy")

with open(auroc_path) as f:
    results = json.load(f)

print("Held-out conversations:", results.get("n_held_out_conversations"))
for layer in results.get("gate_layers", [21, 22]):
    val = results["auroc_by_layer"].get(str(layer), float("nan"))
    target = results.get("paper_targets", {}).get(str(layer))
    print(f"  L{layer}: AUROC={val:.4f}  (paper={target}, gate>={results['gate_threshold']})")

if plot_path.exists():
    display(Image(filename=str(plot_path)))

print(f"\nFrozen axis: {axis_path}  exists={axis_path.exists()}")

## 9. Download artifacts (optional)

Save these back to your machine or keep them on Drive in `stage1/data/`.

In [ ]:
from google.colab import files

for p in [data_file("value_axis.npy"), data_file("auroc_by_layer.json"), data_file("auroc_by_layer.png"), data_file("axis_manifest.json")]:
    if p.exists():
        files.download(str(p))
        print("Downloaded", p.name)

---

**Next steps after gate passes:**
- Axis is frozen at `data/value_axis.npy`
- Downstream de-risking / SWE-bench projection uses this file
- Authors' scripts can read it via `value-axis/common/paths.py` shim

**If gate fails:** see `stage1/README.md` debugging order (boundaries → chat template → hook layer → split → ICRL quality).